# LLM Comprehensive Performance Analysis: Accuracy, Cost & Latency
This notebook performs full evaluation across **Accuracy**, **Cost (USD)**, and **Latency (Seconds)** for LLM models across benchmark classes (`dataset_name`).

### Instructions for Google Colab:
1. Run **Cell 1** to upload `cleaned_datasets.zip` or individual CSV files.
2. Run **Cell 2, 3, 4, & 5** to automatically parse data, generate accuracy, cost, and latency graphs, and view efficiency tradeoff scatter plots!

In [ ]:
# Cell 1: Upload zipped cleaned datasets or CSV files
import os
import zipfile
from google.colab import files

uploaded = files.upload()
uploaded_zip = [f for f in uploaded.keys() if f.endswith('.zip')]

if uploaded_zip:
    zip_path = uploaded_zip[0]
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('extracted_datasets')
    print('Successfully extracted', zip_path)
else:
    os.makedirs('extracted_datasets/individual', exist_ok=True)
    for f in uploaded.keys():
        if f.endswith('.csv'):
            os.rename(f, os.path.join('extracted_datasets/individual', f))
    print('Uploaded individual CSV files.')

In [ ]:
# Cell 2: Accuracy per Model across Benchmark Classes
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate CSV files
csv_files = sorted(glob.glob('extracted_datasets/**/*.csv', recursive=True))
if not csv_files:
    csv_files = sorted(glob.glob('*.csv'))

# Pick aligned_8_models if available, else first valid directory
aligned_8 = [f for f in csv_files if 'aligned_8_models' in f]
if aligned_8:
    target_files = sorted(aligned_8)
else:
    target_files = sorted(csv_files)

os.makedirs('graphs_output', exist_ok=True)
sns.set_theme(style='whitegrid')

print(f'Processing {len(target_files)} evaluation files...')

for f in target_files:
    model_name = os.path.basename(f).replace('.csv', '')
    df = pd.read_csv(f)
    df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0.0)
    
    # Group by dataset_name
    acc_df = df.groupby('dataset_name')['score'].mean().reset_index()
    acc_df['Accuracy (%)'] = acc_df['score'] * 100.0
    acc_df = acc_df.sort_values(by='Accuracy (%)', ascending=False)
    
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(x='dataset_name', y='Accuracy (%)', data=acc_df, palette='viridis', hue='dataset_name', legend=False)
    plt.title(f'Model Accuracy by Benchmark Class: {model_name}', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Benchmark Class', fontsize=12, fontweight='bold')
    plt.ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    plt.ylim(0, 100)
    plt.xticks(rotation=15)
    
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.1f}%', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 3), textcoords='offset points')
    
    plt.tight_layout()
    save_path = f'graphs_output/{model_name}_accuracy.png'
    plt.savefig(save_path, dpi=300)
    plt.show()
    print(f'✓ Graph saved for {model_name} -> {save_path}')

In [ ]:
# Cell 3: Cost and Latency Comparison Graphs
summary_data = []

for f in target_files:
    m_name = os.path.basename(f).replace('.csv', '')
    df = pd.read_csv(f)
    df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0.0)
    df['cost'] = pd.to_numeric(df['cost'], errors='coerce').fillna(0.0)
    df['estimated_latency'] = pd.to_numeric(df['estimated_latency'], errors='coerce').fillna(0.0)
    
    summary_data.append({
        'Model': m_name,
        'Accuracy (%)': df['score'].mean() * 100.0,
        'Avg Cost ($)': df['cost'].mean(),
        'Cost per 1k ($)': df['cost'].mean() * 1000.0,
        'Avg Latency (s)': df['estimated_latency'].mean()
    })

metrics_df = pd.DataFrame(summary_data)

# --- Cost Bar Chart ---
plt.figure(figsize=(10, 5))
cost_sorted = metrics_df.sort_values(by='Avg Cost ($)', ascending=True)
ax1 = sns.barplot(x='Avg Cost ($)', y='Model', data=cost_sorted, palette='Blues_r', hue='Model', legend=False)
plt.title('Average Cost per Query (USD) by Model', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Cost ($)', fontsize=12, fontweight='bold')
plt.ylabel('Model', fontsize=12, fontweight='bold')
for p in ax1.patches:
    width = p.get_width()
    ax1.annotate(f'${width:.5f}', (width, p.get_y() + p.get_height()/2),
                 ha='left', va='center', fontsize=10, fontweight='bold', xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.savefig('graphs_output/model_cost_comparison.png', dpi=300)
plt.show()

# --- Latency Bar Chart ---
plt.figure(figsize=(10, 5))
lat_sorted = metrics_df.sort_values(by='Avg Latency (s)', ascending=True)
ax2 = sns.barplot(x='Avg Latency (s)', y='Model', data=lat_sorted, palette='Oranges_r', hue='Model', legend=False)
plt.title('Average Latency / Response Time (Seconds) by Model', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Average Latency (Seconds)', fontsize=12, fontweight='bold')
plt.ylabel('Model', fontsize=12, fontweight='bold')
for p in ax2.patches:
    width = p.get_width()
    ax2.annotate(f'{width:.3f}s', (width, p.get_y() + p.get_height()/2),
                 ha='left', va='center', fontsize=10, fontweight='bold', xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.savefig('graphs_output/model_latency_comparison.png', dpi=300)
plt.show()

In [ ]:
# Cell 4: Tradeoff Scatter Plots (Accuracy vs. Cost & Accuracy vs. Latency)

# Accuracy vs Cost Plot
plt.figure(figsize=(9, 6))
plt.scatter(metrics_df['Cost per 1k ($)'], metrics_df['Accuracy (%)'], color='#7570b3', s=150, edgecolors='black', zorder=5)
for _, row in metrics_df.iterrows():
    plt.annotate(row['Model'], (row['Cost per 1k ($)'], row['Accuracy (%)']),
                 xytext=(8, -4), textcoords='offset points', fontsize=11, fontweight='bold')
plt.xscale('log')
plt.xlabel('Cost per 1,000 Queries (USD Log-scale)', fontsize=12, fontweight='bold')
plt.ylabel('Mean Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Tradeoff Analysis: Mean Accuracy vs. Cost (USD per 1k Queries)', fontsize=13, fontweight='bold', pad=15)
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('graphs_output/accuracy_vs_cost_tradeoff.png', dpi=300)
plt.show()

# Accuracy vs Latency Plot
plt.figure(figsize=(9, 6))
plt.scatter(metrics_df['Avg Latency (s)'], metrics_df['Accuracy (%)'], color='#1b9e77', s=150, edgecolors='black', zorder=5)
for _, row in metrics_df.iterrows():
    plt.annotate(row['Model'], (row['Avg Latency (s)'], row['Accuracy (%)']),
                 xytext=(8, -4), textcoords='offset points', fontsize=11, fontweight='bold')
plt.xlabel('Average Latency (Seconds)', fontsize=12, fontweight='bold')
plt.ylabel('Mean Accuracy (%)', fontsize=12, fontweight='bold')
plt.title('Tradeoff Analysis: Mean Accuracy vs. Response Latency', fontsize=13, fontweight='bold', pad=15)
plt.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('graphs_output/accuracy_vs_latency_tradeoff.png', dpi=300)
plt.show()

In [ ]:
# Cell 5: Comprehensive Performance Summary Table
print('=== Comprehensive LLM Performance Summary Table ===')
summary_display = metrics_df.sort_values(by='Accuracy (%)', ascending=False).reset_index(drop=True)
display(summary_display.style.format({
    'Accuracy (%)': '{:.2f}%',
    'Avg Cost ($)': '${:.6f}',
    'Cost per 1k ($)': '${:.4f}',
    'Avg Latency (s)': '{:.3f}s'
}).background_gradient(cmap='Greens', subset=['Accuracy (%)']))